## `nz_magabs.ipynb`
-------------

**Perturbed magnification coefficients.**

Run order: `nz.ipynb` (`desi`) → **this notebook** (`desi`) →
`nz_splines_magabs.ipynb` (`pymc`) → `nz_plots_magabs.ipynb` (`pymc`).

In [ ]:
import numpy as np
import importlib
import json
import matplotlib.pyplot as plt
import pandas as pd

from datetime import datetime
from pathlib import Path

import src.statistics.cosmotools as ct
import src.statistics.corrfiles as cf
import src.statistics.inference as inference
import src.statistics.systematics as sy

## autoreload
importlib.reload(ct)
importlib.reload(inference)
importlib.reload(sy)

ROOT = cf.get_base_dir()
CORR_ROOT = ROOT / "src" / "statistics" / "outputs"

In [ ]:
STUDY = "magabs"

scale_cut = [0.3, 3]
version = "v_1p1"

MAG_PERTURBATION = 0.1
N_REALIZATIONS = 100
SEED = 20260823

tomo_to_tracer = sy.TOMO_TO_TRACER
tag = sy.scale_cut_tag(scale_cut)

FID_DIR = ROOT / "results" / "distributions" / f"{tag}_{version}"
OUT_DIR = sy.variant_dir(ROOT, STUDY, scale_cut, version)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Study           : {STUDY}")
print(f"Scale cut       : {scale_cut[0]} - {scale_cut[1]} Mpc/h, version {version}")
print(f"alpha error     : {MAG_PERTURBATION}")
print(f"Realizations    : 0 (unperturbed) + {N_REALIZATIONS} perturbed")
print(f"Fiducial input  : {FID_DIR}")
print(f"Output          : {OUT_DIR}")

In [ ]:
fid = pd.read_parquet(FID_DIR / f"nz_res_{tag}_{version}.parquet")
fid_tracer = (
    fid[fid["tracer"] != "Merged"]
    .dropna(subset=["npz_bs_bp"])
    .sort_values(["tomo_bin", "tracer", "redshift"])
)
print(fid.columns.tolist())
fid_tracer.head()

In [ ]:
# w_dm only depends on the scale cut and the redshift grid, not on alpha, so it is
# computed once per (tracer, grid) and reused by every realization.
wdm_cache = sy.WdmCache(scale_cut)

# sanity check: unperturbed alpha must reproduce the fiducial magnification correction
check_tomo, check_tracer = 3, "LRG"
sub = fid_tracer[
    (fid_tracer.tomo_bin == check_tomo) & (fid_tracer.tracer == check_tracer)
]
zv_check = sub["redshift"].values
npz0, npz0_err = ct.solve_magnification(
    meas=(sub["npz_bs_bp"].values, sub["npz_bs_bp_err"].values),
    tracer=check_tracer,
    tomo_bin=check_tomo,
    scale_cut=scale_cut,
    zvalues=zv_check,
    w_dm_values=wdm_cache.get(zv_check),
)
ref = sub["npz_bs_bp_mag"].values
print("max |rel diff| vs fiducial :", np.abs(npz0 - ref).max() / np.abs(ref).max())
assert np.allclose(npz0, ref), "realization 0 does not reproduce the fiducial n(z)"
assert np.allclose(npz0_err, sub["npz_bs_bp_mag_err"].values)
print("Unperturbed solve reproduces the fiducial magnification correction.")

In [ ]:
alpha_ref = {}
for tomo in sy.TOMO_BINS:
    for tracer in tomo_to_tracer[tomo]:
        zv = fid_tracer[
            (fid_tracer.tomo_bin == tomo) & (fid_tracer.tracer == tracer)
        ]["redshift"].values
        a_p, a_s, _, _ = ct.parametrize_bias(
            tracer=tracer, tomo_bin=tomo, scale_cut=scale_cut,
            wdm=lambda z: np.interp(z, zv, wdm_cache.get(zv)),
        )
        alpha_ref[(tomo, tracer)] = (float(np.mean(a_p(zv))), float(np.mean(a_s(zv))))

print(f"mean alpha over the fitted range, and what {MAG_PERTURBATION} means for it\n")
print(f"{'bin':>4} {'tracer':>10} {'alpha_p':>9} {'alpha_s':>9} "
      f"{'d/alpha_p':>10} {'d/alpha_s':>10}")
for (tomo, tracer), (ap, asx) in alpha_ref.items():
    fp, fs = MAG_PERTURBATION / abs(ap), MAG_PERTURBATION / abs(asx)
    print(f"{tomo:>4} {tracer:>10} {ap:>9.3f} {asx:>9.3f} {fp:>9.1%} {fs:>9.1%}")

In [ ]:
names_var = ["npz_bs_bp", "npz_bs_bp_mag"]
realizations = {}
scales_log = {}

for r in range(N_REALIZATIONS + 1):
    draw = sy.draw_realization_alpha(MAG_PERTURBATION, r, seed=SEED)
    scales_log[str(r)] = draw
    rows = []

    for tomo in sy.TOMO_BINS:
        for tracer in tomo_to_tracer[tomo]:
            sub = fid_tracer[
                (fid_tracer.tomo_bin == tomo) & (fid_tracer.tracer == tracer)
            ]
            if len(sub) == 0:
                print(f"  no fiducial rows for tomo {tomo}, {tracer} -- skipping")
                continue
            zv = sub["redshift"].values
            npz_mag, npz_mag_err = ct.solve_magnification(
                meas=(sub["npz_bs_bp"].values, sub["npz_bs_bp_err"].values),
                tracer=tracer,
                tomo_bin=tomo,
                scale_cut=scale_cut,
                zvalues=zv,
                w_dm_values=wdm_cache.get(zv),
                **sy.alpha_kwargs(draw, tomo, tracer),
            )
            assert not np.any(np.isnan(npz_mag_err))
            for j, z in enumerate(zv):
                rows.append(
                    {
                        "tomo_bin": tomo,
                        "tracer": tracer,
                        "redshift": z,
                        "npz_bs_bp": sub["npz_bs_bp"].values[j],
                        "npz_bs_bp_err": sub["npz_bs_bp_err"].values[j],
                        "npz_bs_bp_mag": npz_mag[j],
                        "npz_bs_bp_mag_err": npz_mag_err[j],
                    }
                )

    df_r = pd.DataFrame(rows)
    merged = sy.merge_over_tracers(df_r, names=names_var)
    norm = sy.normalize_merged(merged, names=names_var)

    df_r.to_parquet(OUT_DIR / f"nz_res_{tag}_{version}_r{r:02d}.parquet", index=False)
    np.savez_compressed(
        OUT_DIR / f"merged_res_norm_{tag}_{version}_r{r:02d}.npz", **norm
    )
    realizations[r] = norm

    ap = ", ".join(f"{draw['p_offset'][t]:+.3f}" for t in sy.TOMO_BINS)
    print(f"realization {r:2d} | alpha_p offsets: [{ap}]")

print(f"\n{len(realizations)} realizations written to {OUT_DIR}")

In [ ]:
metadata = {
    "study": STUDY,
    "description": (
        "Magnification coefficients perturbed as alpha -> alpha + "
        "N(0, MAG_PERTURBATION)"
    ),
    "scale_cuts": scale_cut,
    "version": version,
    "mag_perturbation": MAG_PERTURBATION,
    "n_realizations": N_REALIZATIONS,
    "seed": SEED,
    "patches": sy.PATCHES,
    "tracers_by_tomo": {str(k): v for k, v in tomo_to_tracer.items()},
    "fiducial_input": str(FID_DIR / f"nz_res_{tag}_{version}.parquet"),
    "alpha_draws": scales_log,
    "creation_date": datetime.now().isoformat(),
}
with open(OUT_DIR / f"{STUDY}_metadata_{tag}_{version}.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps({k: v for k, v in metadata.items() if k != "alpha_draws"}, indent=2))

In [ ]:
# how much do the realizations move the measured points around?
bin_colors = {1: "tab:blue", 2: "tab:orange", 3: "tab:red", 4: "tab:cyan"}

fig, axs = plt.subplots(2, 2, figsize=(11, 7), sharex=False)
for tomo, ax in zip(sy.TOMO_BINS, axs.flat):
    z = realizations[0][f"{tomo}/npz_bs_bp_mag_z"]
    fid_nz = realizations[0][f"{tomo}/npz_bs_bp_mag"]
    fid_err = realizations[0][f"{tomo}/npz_bs_bp_mag_err"]

    stack = np.array(
        [realizations[r][f"{tomo}/npz_bs_bp_mag"] for r in range(1, N_REALIZATIONS + 1)]
    )
    for row in stack:
        ax.plot(z, row, color="gray", alpha=0.4, lw=0.8)
    ax.errorbar(
        z, fid_nz, fid_err, fmt="D", ms=4, capsize=3,
        color=bin_colors[tomo], label="unperturbed",
    )
    ax.plot(z, stack.mean(axis=0), color="k", lw=1.2, ls="--", label="realization mean")
    ax.axhline(0, color="k", ls="--", lw=1)
    ax.set_title(f"Bin {tomo}")
    ax.set_xlabel("Redshift")
    ax.set_ylabel("n(z)")
    ax.grid(True, alpha=0.3)
    if tomo == 1:
        ax.legend(fontsize=9)

fig.suptitle(f"absolute error sigma_alpha = 0.1 ({N_REALIZATIONS} realizations)")
fig.tight_layout()

In [ ]:
# realization scatter compared with the statistical error bar, point by point.
# This is the pre-spline view of how much the systematic can matter.
for tomo in sy.TOMO_BINS:
    z = realizations[0][f"{tomo}/npz_bs_bp_mag_z"]
    fid_err = realizations[0][f"{tomo}/npz_bs_bp_mag_err"]
    stack = np.array(
        [realizations[r][f"{tomo}/npz_bs_bp_mag"] for r in range(1, N_REALIZATIONS + 1)]
    )
    ratio = stack.std(axis=0) / fid_err
    print(
        f"bin {tomo}: realization scatter / statistical error -- "
        f"median {np.median(ratio):.3f}, max {ratio.max():.3f}"
    )